In [2]:
import pandas as pd
import numpy as np
import re
import os
import json
from collections import Counter, defaultdict
from tqdm import tqdm
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

print("All imports successful")

All imports successful


In [3]:
config = TemplateMinerConfig()
config.profiling_enabled = False

template_miner = TemplateMiner(config=config)

print("Drain3 configured and ready")

Drain3 configured and ready


In [9]:
def preprocess_message(message):
    """Mask variable parts so Drain3 can find the real pattern."""
    msg = re.sub(r'blk_-?\d+', '<BLK>', message)
    msg = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '<IP>', msg)
    msg = re.sub(r':\d{4,5}', ':<PORT>', msg)
    msg = re.sub(r'/[\w/\.\-_]+', '<PATH>', msg)
    msg = re.sub(r'\b\d+\b', '<NUM>', msg)
    return msg

test_msg = "Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010"
print(f"RAW:   {test_msg}")
print(f"CLEAN: {preprocess_message(test_msg)}")

RAW:   Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
CLEAN: Receiving block <BLK> src: /<IP>:<PORT> dest: /<IP>:<PORT>


In [10]:
template_miner = TemplateMiner(config=config)

LOG_PATH = "../data/raw/HDFS.log"
results = []

print("Parsing 100,000 lines with preprocessing...")
with open(LOG_PATH, 'r') as f:
    for i, line in enumerate(tqdm(f, total=100_000)):
        if i >= 100_000:
            break
        
        line = line.strip()
        if not line:
            continue
        
        match = re.match(r'\d{6}\s+\d{6}\s+\d+\s+\w+\s+[\w\.\$]+:\s*(.*)', line)
        if not match:
            continue
        
        raw_message = match.group(1)
        clean_message = preprocess_message(raw_message)
        
        block_match = re.search(r'(blk_-?\d+)', raw_message)
        block_id = block_match.group(1) if block_match else None
        
        result = template_miner.add_log_message(clean_message)
        
        results.append({
            'block_id': block_id,
            'event_id': result['cluster_id'],
            'template': result['template_mined'],
        })

df_parsed = pd.DataFrame(results)
print(f"\nParsed: {len(df_parsed):,} lines")
print(f"Templates found: {df_parsed['event_id'].nunique()}")

Parsing 100,000 lines with preprocessing...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100000/100000 [00:02<00:00, 37432.70it/s]



Parsed: 100,000 lines
Templates found: 20


In [11]:
templates = df_parsed.groupby('event_id').agg(
    count=('template', 'size'),
    template=('template', 'first'),
).sort_values('count', ascending=False).reset_index()

print(f"{'ID':>4}  {'Count':>8}  Template")
print("-" * 90)
for _, row in templates.iterrows():
    print(f"{row['event_id']:>4}  {row['count']:>8,}  {row['template']}")

  ID     Count  Template
------------------------------------------------------------------------------------------
   1    23,611  Receiving block <BLK> src: /<IP>:<PORT> dest: /<IP>:<PORT>
   5    22,287  BLOCK* NameSystem.addStoredBlock: blockMap updated: <IP>:<PORT> is added to <BLK> size <NUM>
   4    22,225  Received block <BLK> of size <NUM> from /<IP>
   3    22,224  PacketResponder <NUM> for block <BLK> terminating
   2     7,940  BLOCK* NameSystem.allocateBlock: <PATH> <BLK>
  13     1,088  Verification succeeded for <BLK>
  10       407  <IP>:<PORT> Served block <BLK> to /<IP>
  16        62  writeBlock <BLK> received exception java.io.IOException: Could not read from stream
   6        29  Received block <BLK> src: /<IP>:<PORT> dest: /<IP>:<PORT> of size <NUM>
   7        25  <IP>:<PORT>:Transmitted block <BLK> to /<IP>:<PORT>
  18        24  Receiving empty packet for block <BLK>
  12        22  <IP>:<PORT> Starting thread to transfer block <BLK> to <IP>:<PORT>
  11       

In [12]:
sequences = df_parsed.groupby('block_id')['event_id'].apply(list).reset_index()
sequences.columns = ['block_id', 'sequence']
sequences['seq_length'] = sequences['sequence'].apply(len)

print(f"Total blocks: {len(sequences):,}")
print(f"\nSequence length stats:")
print(sequences['seq_length'].describe().round(1))

print(f"\n--- Example sequences ---")
for i in range(5):
    row = sequences.iloc[i]
    print(f"\n  Block: {row['block_id']}")
    print(f"  Length: {row['seq_length']}")
    print(f"  Sequence: {row['sequence']}")

Total blocks: 7,940

Sequence length stats:
count    7940.0
mean       12.6
std         4.3
min         1.0
25%        13.0
50%        13.0
75%        13.0
max       249.0
Name: seq_length, dtype: float64

--- Example sequences ---

  Block: blk_-1001553972418305662
  Length: 13
  Sequence: [2, 1, 1, 1, 5, 5, 3, 4, 3, 4, 5, 3, 4]

  Block: blk_-1010952805175971965
  Length: 13
  Sequence: [2, 1, 1, 1, 5, 5, 5, 3, 4, 3, 4, 3, 4]

  Block: blk_-1011482868748761910
  Length: 13
  Sequence: [2, 1, 1, 1, 5, 5, 3, 4, 3, 4, 5, 3, 4]

  Block: blk_-1011537904811654030
  Length: 13
  Sequence: [1, 2, 1, 1, 3, 4, 3, 4, 3, 4, 5, 5, 5]

  Block: blk_-1015291919896450721
  Length: 13
  Sequence: [2, 1, 1, 1, 5, 4, 3, 4, 5, 5, 3, 3, 4]


In [13]:
labels = pd.read_csv("../data/raw/anomaly_label.csv")
labels.columns = ['block_id', 'label']

seq_labelled = sequences.merge(labels, on='block_id', how='inner')

print(f"Blocks with labels: {len(seq_labelled):,}")
print(f"\n--- Label distribution ---")
print(seq_labelled['label'].value_counts())

print(f"\n--- Normal sequence example ---")
normal = seq_labelled[seq_labelled['label'] == 'Normal'].iloc[0]
print(f"  {normal['sequence']}")

print(f"\n--- Anomaly sequence example ---")
anomaly = seq_labelled[seq_labelled['label'] == 'Anomaly']
if len(anomaly) > 0:
    anom = anomaly.iloc[0]
    print(f"  {anom['sequence']}")
    print(f"  Length: {anom['seq_length']}")
else:
    print("  No anomalies in this sample — need to parse more data")

Blocks with labels: 7,940

--- Label distribution ---
label
Normal     7627
Anomaly     313
Name: count, dtype: int64

--- Normal sequence example ---
  [2, 1, 1, 1, 5, 5, 3, 4, 3, 4, 5, 3, 4]

--- Anomaly sequence example ---
  [1, 1, 1, 2, 3, 4, 3, 4, 3, 4, 5, 5, 5]
  Length: 13


In [14]:
template_miner_full = TemplateMiner(config=config)

LOG_PATH = "../data/raw/HDFS.log"
full_results = []

total_lines = sum(1 for _ in open(LOG_PATH, 'r'))
print(f"Total lines in file: {total_lines:,}")

print(f"\nParsing full dataset...")
with open(LOG_PATH, 'r') as f:
    for i, line in enumerate(tqdm(f, total=total_lines)):
        line = line.strip()
        if not line:
            continue
        
        match = re.match(r'\d{6}\s+\d{6}\s+\d+\s+\w+\s+[\w\.\$]+:\s*(.*)', line)
        if not match:
            continue
        
        raw_message = match.group(1)
        clean_message = preprocess_message(raw_message)
        
        block_match = re.search(r'(blk_-?\d+)', raw_message)
        block_id = block_match.group(1) if block_match else None
        
        result = template_miner_full.add_log_message(clean_message)
        
        full_results.append({
            'block_id': block_id,
            'event_id': result['cluster_id'],
        })

df_full = pd.DataFrame(full_results)
print(f"\nParsed: {len(df_full):,} lines")
print(f"Templates: {df_full['event_id'].nunique()}")

Total lines in file: 11,175,629

Parsing full dataset...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11175629/11175629 [06:04<00:00, 30694.12it/s]



Parsed: 11,175,629 lines
Templates: 45
